# 06 · Simulación what-if (SimPy)
Calibrar con el dato real → probar escenarios → cuantificar mejoras. **Supuesto:** capacidad por etapa = ocupación máxima observada (reemplazar por capacidad real si el hospital la entrega).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, load, quality, flow, features, models

# Fuente de datos:
#   "db"     -> base de datos del reto (FortiClient VPN conectada + archivo .env)
#   "raw"    -> archivos exportados en data/raw
#   "sample" -> datos sintéticos de prueba (scripts/generar_datos_sinteticos.py)
SOURCE = "db"
DATA_DIR = ROOT / "data" / ("sample" if SOURCE == "sample" else "raw")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
if SOURCE == "db":
    from src import db
    db.test_connection()
    t = db.load_all()   # copia local en data/interim; db.load_all(refresh=True) para volver a descargar
else:
    t = load.load_all(DATA_DIR)

In [ ]:
from src import simulation as sim
ev = pd.read_parquet(config.DATA_INTERIM / 'event_log.parquet')
rate, routes, probs, service = sim.fit_inputs(ev, t['visit_occurrence'])
cap_max = sim.estimate_capacity(ev)
cap_max

## Calibración: capacidad efectiva que reproduce las esperas reales
La espera real entre etapas (`espera_previa_h`) se usa como objetivo. Si el hospital entrega la capacidad real (camas, consultorios, personal por turno), usarla en vez de la calibrada.

In [ ]:
real_wait = ev.groupby('etapa')['espera_previa_h'].mean()
cap, calib = sim.calibrate(rate, routes, probs, service, cap_max, real_wait, days=30)
calib

In [ ]:
base = sim.run_reps(rate, routes, probs, service, cap, days=60, reps=10)
simu = base.groupby('etapa')['espera_h'].mean().rename('sim_media_h')
pd.concat([real_wait.rename('real_media_h'), simu], axis=1).round(2)

## Escenario: +1 recurso en la etapa cuello de botella

In [ ]:
cuello = base.groupby('etapa')['espera_h'].mean().idxmax()
print('Cuello de botella:', cuello)
esc = sim.run_reps(rate, routes, probs, service, cap, days=60, reps=10, extra_capacity={cuello: 1})
comp = sim.compare(base, esc)
comp.round(2)

In [ ]:
comp.to_csv(config.DATA_PROCESSED / 'simulacion_escenarios.csv')

## Recomendación
- _(ej.: 'Agregar 1 recurso en X reduce la espera media de Y h a Z h (−N%)')_